In [155]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Lasso

In [156]:
train = pd.read_csv('data/train.csv')

x_test = pd.read_csv('data/test.csv')
# y_train = train['SalePrice']
y_train = np.log1p(train['SalePrice'])
x_train = train.drop(columns=['SalePrice'])

In [157]:
for df in [x_train, x_test]:
    df['HasPool'] = df['PoolQC'].notnull().astype(int)
    df['HasFireplace'] = df['FireplaceQu'].notnull().astype(int) # камины тоже важны!

# Описываем «Белый список» — эти столбцы робот-фильтр трогать не имеет права
whitelist = ['HasPool', 'HasFireplace', 'OverallQual', 'GrLivArea']

In [158]:
test_ids = x_test['Id']

In [159]:
MISSING_THRESHOLD = 80.0
IMBALANCE_THRESHOLD = 95.0

drop_cols = []
for col in x_train.columns:
    if col in whitelist:
        continue

    if col.lower() == 'id':
        drop_cols.append(col)

    missing = x_train[col].isnull().mean() * 100
    if missing > MISSING_THRESHOLD:
        drop_cols.append(col)
        continue

    imbalance = x_train[col].value_counts(normalize=True).values[0] * 100
    if imbalance > IMBALANCE_THRESHOLD:
        drop_cols.append(col)
        continue

In [160]:
print(f"Найдено {len(drop_cols)} плохих столбцов: {drop_cols}")

x_train = x_train.drop(columns=drop_cols)
x_test = x_test.drop(columns=drop_cols)

Найдено 17 плохих столбцов: ['Id', 'Street', 'Alley', 'Utilities', 'Condition2', 'RoofMatl', 'Heating', 'LowQualFinSF', 'KitchenAbvGr', 'GarageQual', 'GarageCond', '3SsnPorch', 'PoolArea', 'PoolQC', 'Fence', 'MiscFeature', 'MiscVal']


In [161]:
num_col_head = x_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_col_head = x_train.select_dtypes(include=['object', 'str']).columns.tolist()

In [162]:
num_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transform = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

In [163]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transform, num_col_head),
        ('cat', cat_transform, cat_col_head)
    ]
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=10000)) # увеличиваем итерации для сходимости
])

In [164]:
param_grid = {
    'model__alpha': [0.0001, 0.0003, 0.0005, 0.001, 0.005, 0.01, 0.1]
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(x_train, y_train)
print(f"Лучший параметр alpha: {grid_search.best_params_['model__alpha']}")

Лучший параметр alpha: 0.0005


In [165]:
log_predictions = grid_search.predict(x_test)

# Возвращаем цены из логарифмического масштаба в реальные доллары
final_predictions = np.expm1(log_predictions)

C:\Users\delux\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 10, 11, 24, 25, 30] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [166]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})
submission.to_csv('submission_lasso.csv', index=False)
print("Файл submission_lasso.csv успешно создан и готов к отправке!")

Файл submission_lasso.csv успешно создан и готов к отправке!
